# 5 Advanced Visualizations

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from ift6758.features.simple_visualizations import games_list

In [3]:
def number_games(season) -> dict:
    # Get all the games dataframe
    df_games = games_list(season, events=False)

    # Keep regular season and playoffs only (game type code in the game id)
    df_games = df_games[df_games["game_type"].isin([2, 3])]

    # Calculate the number of game for each team
    number_games_per_team = {}
    for game in df_games.itertuples():
        homeTeam_id = getattr(game, "homeTeam_name")
        awayTeam_id = getattr(game, "awayTeam_name")
        number_games_per_team[homeTeam_id] = number_games_per_team.get(homeTeam_id, 0) + 1
        number_games_per_team[awayTeam_id] = number_games_per_team.get(awayTeam_id, 0) + 1

    return number_games_per_team

print(number_games(2022))

{'Nashville_Predators': 82, 'San Jose_Sharks': 82, 'New York_Rangers': 89, 'Tampa Bay_Lightning': 88, 'Los Angeles_Kings': 88, 'Vegas_Golden Knights': 104, 'Washington_Capitals': 82, 'Boston_Bruins': 89, 'Carolina_Hurricanes': 97, 'Columbus_Blue Jackets': 82, 'Montréal_Canadiens': 82, 'Toronto_Maple Leafs': 93, 'Colorado_Avalanche': 89, 'Chicago_Blackhawks': 82, 'Anaheim_Ducks': 82, 'Seattle_Kraken': 96, 'Edmonton_Oilers': 94, 'Vancouver_Canucks': 82, 'Pittsburgh_Penguins': 82, 'Arizona_Coyotes': 82, 'Philadelphia_Flyers': 82, 'New Jersey_Devils': 94, 'Buffalo_Sabres': 82, 'Ottawa_Senators': 82, 'New York_Islanders': 88, 'Florida_Panthers': 103, 'Dallas_Stars': 101, 'Minnesota_Wild': 88, 'Calgary_Flames': 82, 'Detroit_Red Wings': 82, 'Winnipeg_Jets': 87, 'St. Louis_Blues': 82}


In [ ]:

def shot_data(season):
    """
    Gets the data from json (possibly by calling the distance function) for the season and extracts the shot coordinates to a more usable format
    :param season: 4 digit year of the season start
    :return: a dataframe containing each team's name, number of games, and an array of x,y shot coordinates (1 line per shot, columns x and y)
    """
    # Get each team's number of games
    team_game_number = number_games(season)

    # Link each team_id to its abbrev
    df_games = games_list(season, events=False)
    id_to_abbrev = {}
    for game in df_games.itertuples():
        id_to_abbrev[getattr(game, "homeTeam_id")] = getattr(game, "homeTeam_name")
        id_to_abbrev[getattr(game, "awayTeam_id")] = getattr(game, "awayTeam_name")

    # Get the shots and goals dataframe for the given season
    df_events = games_list(season)

    # Keep regular season and playoffs
    df_shots = df_events[
        df_events["game_type"].isin(["regular season", "playoffs"])
    ].copy()

    # Put all the shots on the same side (use same logic as distance() from simple_visualization.py)
    home_shoots = df_shots["team_id"] == df_shots["homeTeam_id"]
    attacks_left = np.where(
        home_shoots,
        df_shots["homeTeamDefendingSide"] == "right",
        df_shots["homeTeamDefendingSide"] == "left"
    )

    # Flip the shots by 180°­­­­ if they were taken from the left side of the pitch
    df_shots.loc[attacks_left, ["x_coord", "y_coord"]] *= -1

    rows = []
    for team_id, df_team in df_shots.groupby("team_id"):
        team = id_to_abbrev[team_id]
        rows.append({
            "team": team,
            "games": team_game_number[team],
            "shots": df_team[["x_coord", "y_coord"]].T
        })

    return pd.DataFrame(rows)

print(shot_data(2022).head())

In [ ]:

def kde():
    """
    Calculates kde when given an array of shot coordinates
    :return: shot density
    """



In [ ]:

class ShotMap:

    def __init__(self, season):
        """

        :param season: 4 digit year of the season start
        """
        self.season = season
        self.shots = shot_data(season)
        self.kdes = []

    def calculate_kdes(self):
        """
        This calculates league average shot rate per hour (league_sph) and excess shot rate per hour for each team, in the form of shot density using kde. Also saves raw difference in shots per team hour
        :return:
        """
        # first, calculate kde for league average and save the factor, which will be given to the team kdes

        # calculate shot rate per hour density for each team using kde:
        for team_coords in self.shots["shots"]:
            print(team_coords)
            self.kdes.append(stats.gaussian_kde(team_coords))
        # densities will need to be weighed by shot rate

        # convert teams' sph to excess sph using league average
        # calculate raw difference in shots per team hour


In [ ]:
shots2020 = ShotMap(2020)


In [ ]:
print(shots2020.shots.shape)
print(shots2020.shots.columns)


In [ ]:
shots2020.calculate_kdes()
shots2020.shots.head()
print(shots2020.kdes)

In [ ]:
print(shots2020.kdes[0])
X, Y = np.mgrid[-100:100:200j, -100:100:200j]
positions = np.vstack([X.ravel(), Y.ravel()])
# différence entre deux équipes:
plt.imshow(np.reshape(shots2020.kdes[0](positions).T, X.shape) - np.reshape(shots2020.kdes[1](positions).T, X.shape), cmap = "gnuplot2")
plt.show()

In [ ]:
def interactive_shot_map(shot_map):
    """
    :param shot_map: ShotMap object
    :return: Creates html interactive figure ?
    """
    # checks if shot_map has kdes, if not, calls shot_map.calculate_kde()

In [ ]:
interactive_shot_map(shots2020)